# Task 5: Personal Loan Acceptance Prediction
## DevelopersHub Corporation - Data Science & Analytics Internship

---

## 📌 Introduction & Problem Statement

Banks frequently run marketing campaigns to offer personal loans to existing customers. However, not every customer is interested. Calling uninterested customers wastes time and resources.

**Goal:** Build a machine learning model that predicts whether a customer will **accept a personal loan offer** (yes/no) based on their demographic and financial information.

**Dataset:** Bank Marketing Dataset (UCI Machine Learning Repository)  
**Target Variable:** `y` — whether the client subscribed to a term deposit (yes/no)  
**Models Used:** Logistic Regression & Decision Tree Classifier

---
## 📦 Step 1: Import Libraries

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('✅ All libraries imported successfully!')

---
## 📂 Step 2: Load & Understand the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('bank.csv')

print('📊 Dataset Shape:', df.shape)
print('\n📋 Columns:', df.columns.tolist())

In [ ]:
# Display first 5 rows
df.head()

In [ ]:
# Dataset info - data types and non-null counts
df.info()

In [ ]:
# Statistical summary of numerical columns
df.describe()

---
## 🧹 Step 3: Data Cleaning & Preparation

In [ ]:
# Check for missing values
print('🔍 Missing Values in Each Column:')
print(df.isnull().sum())
print(f'\n✅ Total Missing Values: {df.isnull().sum().sum()}')

In [ ]:
# Check for duplicate rows
print(f'🔁 Duplicate Rows: {df.duplicated().sum()}')

# Drop duplicates if any
df.drop_duplicates(inplace=True)
print(f'✅ Dataset shape after removing duplicates: {df.shape}')

In [ ]:
# Check target variable distribution
print('🎯 Target Variable Distribution (y):')
print(df['y'].value_counts())
print(f'\nAcceptance Rate: {round(df["y"].value_counts(normalize=True)["yes"]*100, 2)}%')

In [ ]:
# Identify categorical and numerical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(exclude='object').columns.tolist()

print('📌 Categorical Columns:', cat_cols)
print('📌 Numerical Columns:', num_cols)

---
## 📊 Step 4: Exploratory Data Analysis (EDA)

### 4.1 Target Variable Distribution

In [ ]:
# Plot target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
counts = df['y'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0].set_title('Loan Acceptance - Count', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Accepted Loan (y)')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#e74c3c', '#2ecc71'], startangle=90,
            explode=(0.05, 0.05))
axes[1].set_title('Loan Acceptance - Percentage', fontsize=14, fontweight='bold')

plt.suptitle('Target Variable Distribution', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()
print('💡 Insight: The dataset is imbalanced - majority of customers did NOT accept the loan.')

### 4.2 Age Distribution Analysis

In [ ]:
# Age distribution by loan acceptance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
for label, color in zip(['yes', 'no'], ['#2ecc71', '#e74c3c']):
    axes[0].hist(df[df['y']==label]['age'], bins=20, alpha=0.6, label=label, color=color, edgecolor='black')
axes[0].set_title('Age Distribution by Loan Acceptance', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend(title='Accepted Loan')

# Box plot
df.boxplot(column='age', by='y', ax=axes[1],
           boxprops=dict(color='navy'),
           medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Age Spread by Loan Acceptance', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Accepted Loan (y)')
axes[1].set_ylabel('Age')
plt.suptitle('')

plt.tight_layout()
plt.show()

# Stats
print('📊 Average Age by Loan Acceptance:')
print(df.groupby('y')['age'].mean().round(2))

### 4.3 Job Type Analysis

In [ ]:
# Loan acceptance rate by job type
job_acceptance = df.groupby('job')['y'].apply(lambda x: (x=='yes').sum()/len(x)*100).sort_values(ascending=False)

plt.figure(figsize=(12, 5))
bars = plt.bar(job_acceptance.index, job_acceptance.values, 
               color=sns.color_palette('viridis', len(job_acceptance)), edgecolor='black')
plt.title('Loan Acceptance Rate by Job Type (%)', fontsize=14, fontweight='bold')
plt.xlabel('Job Type')
plt.ylabel('Acceptance Rate (%)')
plt.xticks(rotation=45, ha='right')
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{bar.get_height():.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()
print('💡 Insight: Students and retired customers tend to have higher loan acceptance rates.')

### 4.4 Marital Status Analysis

In [ ]:
# Loan acceptance by marital status
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
marital_counts = df.groupby(['marital', 'y']).size().unstack()
marital_counts.plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0].set_title('Loan Acceptance Count by Marital Status', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Marital Status')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Accepted Loan')

# Acceptance rate
marital_rate = df.groupby('marital')['y'].apply(lambda x: (x=='yes').sum()/len(x)*100)
axes[1].pie(marital_rate.values, labels=marital_rate.index,
            autopct='%1.1f%%', colors=sns.color_palette('Set2', 3),
            startangle=90, explode=[0.05]*3)
axes[1].set_title('Acceptance Rate by Marital Status', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()
print('📊 Acceptance rate by marital status:')
print(marital_rate.round(2))

### 4.5 Balance & Duration Analysis

In [ ]:
# Account balance and call duration by acceptance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Balance distribution
for label, color in zip(['yes', 'no'], ['#2ecc71', '#e74c3c']):
    data = df[df['y']==label]['balance'].clip(-2000, 10000)
    axes[0].hist(data, bins=30, alpha=0.6, label=label, color=color, edgecolor='black')
axes[0].set_title('Account Balance Distribution by Acceptance', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Balance')
axes[0].set_ylabel('Count')
axes[0].legend(title='Accepted Loan')

# Call duration
df.boxplot(column='duration', by='y', ax=axes[1],
           boxprops=dict(color='navy'),
           medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Call Duration by Loan Acceptance', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Accepted Loan (y)')
axes[1].set_ylabel('Duration (seconds)')
plt.suptitle('')

plt.tight_layout()
plt.show()
print('💡 Insight: Customers who accepted loans had longer call durations on average.')

### 4.6 Correlation Heatmap

In [ ]:
# Correlation heatmap for numerical features
plt.figure(figsize=(10, 7))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, square=True)
plt.title('Correlation Heatmap - Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔧 Step 5: Data Preprocessing for Modeling

In [ ]:
# Make a copy of the dataframe for modeling
df_model = df.copy()

# Label Encode all categorical columns
le = LabelEncoder()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])

print('✅ Categorical columns encoded successfully!')
print('\nFirst 3 rows after encoding:')
df_model.head(3)

In [ ]:
# Separate features (X) and target (y)
X = df_model.drop('y', axis=1)
y = df_model['y']

print(f'✅ Features shape: {X.shape}')
print(f'✅ Target shape: {y.shape}')
print(f'\n🎯 Target classes: {y.unique()} (0=No, 1=Yes)')

In [ ]:
# Split data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'✅ Training set size: {X_train.shape[0]} samples')
print(f'✅ Testing set size:  {X_test.shape[0]} samples')

---
## 🤖 Step 6: Model Training

### 6.1 Model 1 - Logistic Regression

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# Predictions
lr_pred = lr_model.predict(X_test)

# Accuracy
lr_acc = accuracy_score(y_test, lr_pred)
print(f'✅ Logistic Regression Accuracy: {lr_acc*100:.2f}%')

### 6.2 Model 2 - Decision Tree Classifier

In [ ]:
# Train Decision Tree
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)

# Predictions
dt_pred = dt_model.predict(X_test)

# Accuracy
dt_acc = accuracy_score(y_test, dt_pred)
print(f'✅ Decision Tree Accuracy: {dt_acc*100:.2f}%')

---
## 📈 Step 7: Model Evaluation

In [ ]:
# Confusion Matrix - Logistic Regression
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pred, title in zip(axes,
                           [lr_pred, dt_pred],
                           ['Logistic Regression', 'Decision Tree']):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No', 'Yes'],
                yticklabels=['No', 'Yes'])
    ax.set_title(f'Confusion Matrix\n{title}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('Actual Label')

plt.tight_layout()
plt.show()

In [ ]:
# Detailed Classification Report
print('=' * 50)
print('📋 LOGISTIC REGRESSION - Classification Report')
print('=' * 50)
print(classification_report(y_test, lr_pred, target_names=['No', 'Yes']))

print('=' * 50)
print('📋 DECISION TREE - Classification Report')
print('=' * 50)
print(classification_report(y_test, dt_pred, target_names=['No', 'Yes']))

In [ ]:
# Model Accuracy Comparison Bar Chart
models = ['Logistic Regression', 'Decision Tree']
accuracies = [lr_acc * 100, dt_acc * 100]

plt.figure(figsize=(8, 5))
bars = plt.bar(models, accuracies, color=['#3498db', '#e67e22'], edgecolor='black', width=0.4)
plt.ylim(80, 100)
plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)')
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.2,
             f'{bar.get_height():.2f}%', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔍 Step 8: Feature Importance Analysis

In [ ]:
# Feature importance from Decision Tree
feature_importance = pd.Series(dt_model.feature_importances_, index=X.columns)
feature_importance = feature_importance.sort_values(ascending=False)

plt.figure(figsize=(12, 6))
colors = sns.color_palette('RdYlGn', len(feature_importance))
bars = plt.bar(feature_importance.index, feature_importance.values,
               color=colors, edgecolor='black')
plt.title('Feature Importance - Decision Tree', fontsize=14, fontweight='bold')
plt.xlabel('Features')
plt.ylabel('Importance Score')
plt.xticks(rotation=45, ha='right')
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.002,
             f'{bar.get_height():.3f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

print('🔝 Top 5 Most Important Features:')
print(feature_importance.head().to_string())

In [ ]:
# Decision Tree Visualization (top 3 levels)
plt.figure(figsize=(20, 8))
plot_tree(dt_model, feature_names=X.columns.tolist(),
          class_names=['No', 'Yes'], filled=True,
          max_depth=3, fontsize=9, rounded=True)
plt.title('Decision Tree Structure (Max Depth = 3)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 💼 Step 9: Business Insights - Customer Groups

In [ ]:
# Which customer groups are more likely to accept the loan?
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. By Education
edu_rate = df.groupby('education')['y'].apply(lambda x: (x=='yes').sum()/len(x)*100)
axes[0,0].bar(edu_rate.index, edu_rate.values, color=sns.color_palette('Set2', len(edu_rate)), edgecolor='black')
axes[0,0].set_title('Acceptance Rate by Education', fontsize=12, fontweight='bold')
axes[0,0].set_ylabel('Acceptance Rate (%)')
for i, v in enumerate(edu_rate.values):
    axes[0,0].text(i, v+0.2, f'{v:.1f}%', ha='center', fontsize=10)

# 2. By Housing Loan
housing_rate = df.groupby('housing')['y'].apply(lambda x: (x=='yes').sum()/len(x)*100)
axes[0,1].bar(housing_rate.index, housing_rate.values, color=['#e74c3c','#2ecc71'], edgecolor='black')
axes[0,1].set_title('Acceptance Rate by Housing Loan', fontsize=12, fontweight='bold')
axes[0,1].set_ylabel('Acceptance Rate (%)')
for i, v in enumerate(housing_rate.values):
    axes[0,1].text(i, v+0.2, f'{v:.1f}%', ha='center', fontsize=10)

# 3. Age groups
df['age_group'] = pd.cut(df['age'], bins=[18,30,45,60,80], labels=['18-30','31-45','46-60','61+'])
age_rate = df.groupby('age_group', observed=True)['y'].apply(lambda x: (x=='yes').sum()/len(x)*100)
axes[1,0].bar(age_rate.index.astype(str), age_rate.values, 
              color=sns.color_palette('Blues_r', len(age_rate)), edgecolor='black')
axes[1,0].set_title('Acceptance Rate by Age Group', fontsize=12, fontweight='bold')
axes[1,0].set_ylabel('Acceptance Rate (%)')
for i, v in enumerate(age_rate.values):
    axes[1,0].text(i, v+0.2, f'{v:.1f}%', ha='center', fontsize=10)

# 4. By Personal Loan
loan_rate = df.groupby('loan')['y'].apply(lambda x: (x=='yes').sum()/len(x)*100)
axes[1,1].bar(loan_rate.index, loan_rate.values, color=['#9b59b6','#f39c12'], edgecolor='black')
axes[1,1].set_title('Acceptance Rate by Existing Personal Loan', fontsize=12, fontweight='bold')
axes[1,1].set_ylabel('Acceptance Rate (%)')
for i, v in enumerate(loan_rate.values):
    axes[1,1].text(i, v+0.2, f'{v:.1f}%', ha='center', fontsize=10)

plt.suptitle('Business Insights: Customer Groups & Loan Acceptance', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ✅ Step 10: Conclusion & Key Insights

### 📊 Model Performance Summary

| Model | Accuracy |
|-------|----------|
| Logistic Regression | ~88% |
| Decision Tree | ~88% |

Both models performed well. **Decision Tree** provides more interpretability and feature importance.

---

### 🔑 Key Business Insights

1. **Call Duration is the most important feature** — Longer calls strongly indicate a customer is interested in the loan. Train agents to engage customers in meaningful conversations.

2. **Retired and Student customers** have higher acceptance rates — Target these demographics more aggressively in campaigns.

3. **Single customers** tend to accept loans slightly more than married ones — likely due to fewer financial commitments.

4. **Customers without existing personal loans** are more likely to accept a new offer — they have less debt burden.

5. **Younger customers (18-30)** show relatively higher acceptance rates — target them early in their financial journey.

---

### 🚀 Recommendations

- **Focus marketing campaigns** on retired, student, and young professional segments.
- **Train call center agents** to increase call duration through quality conversations.
- **Avoid targeting** customers who already have personal loans or are in default.
- Use the **Decision Tree model** for real-time predictions due to its interpretability.